<a href="https://colab.research.google.com/github/spicecat/Haxballers/blob/main/Haxballers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Haxballers


TODO:
- improve reward function
- retrain model across drills
    - add drills with StateSetters
- hyperparameter tuning
- add comments/documentation

[https://drive.google.com/drive/folders/1HyP6M42N6AX-f2QrKNgDnpI86EC5KBK2](https://drive.google.com/drive/folders/1HyP6M42N6AX-f2QrKNgDnpI86EC5KBK2)

## Summary

Our project idea is to develop and train a multi-agent system to play the game of Haxball, a simplified soccer simulation game. The behaviors we want the agents to learn is to score goals, defend against goals, and pass to teammates. Agents will use the current game state as input, including the positions and velocities of all players and the ball, and output a cardinal direction to move in or a kick action.

## Project Goals

- Minimum Goal: Learn to use gymnasium and stablebaseline to develop and train an agent that can move to the ball.
- Realistic Goal: Implement drills for agents to practice scoring and defending goals and attempt hyperparameter optimization.
- Moonshot Goal: Implement drills for agents to coordination with teammates, custom observation builders and reward functions, and a system to optimize hyperparameters.

## Algorithms

We anticipate using model-free on-policy multi-agent reinforcement learning to train the Haxball agents. The training environment will progress through the following stages: 1v0, 1v1, 2v1, 2v2, and 3v3.

## Evaluation Plan

For quantitative evaluation, we will have agents at different levels of training compete in an Elo rating system. Some metrics we may measure are the number of games won, the number of goals scored, the number of goals defended, and pass frequency. As a baseline approach, we will train agents to move behind the ball and kick it towards the opposing goal. We estimate that successful training will improve the win rate metric by 90%.

For qualitative analysis, we will use the 1v0 training environment for sanity checks. We will visualize the results externally by reviewing game replays. A successful result is expected to display agents moving efficiently, kicking the ball towards the opposing goal, and passing the ball towards open teammates.

## AI Usage

AI was used for coding assistance.


## Setup

Restart session after installing libraries to fix `ModuleNotFoundError: No module named 'haxballgym'`

In [ ]:
# @title Install Libraries
!git clone --recursive -q https://github.com/spicecat/HaxballGym.git
%pip install -e HaxballGym > /dev/null
%pip install gymnasium openskill optuna pettingzoo pyvirtualdisplay stable-baselines3 supersuit tensorboard -q

In [1]:
# @title Import Libraries
import copy
import multiprocessing
import shutil
import subprocess
import warnings
from concurrent.futures import ProcessPoolExecutor

import haxballgym
import numpy as np
import numpy.typing as npt
import supersuit as ss
from haxballgym.utils.gamestates import GameState as HBGGameState
from haxballgym.utils.obs_builders import DefaultObs, ObsBuilder
from haxballgym.utils.reward_functions import CombinedReward, RewardFunction
from haxballgym.utils.reward_functions.common_rewards import EventReward
from haxballgym.utils.reward_functions.velocity_reward import (
    VelocityBallToGoalReward,
    VelocityPlayerToBallReward,
)
from haxballgym.utils.state_setters import DefaultState, RandomState, StateSetter
from haxballgym.utils.terminal_conditions import TerminalCondition, common_conditions
from IPython.display import Video
from openskill.models import PlackettLuce, PlackettLuceRating
from pettingzoo import ParallelEnv
from pyvirtualdisplay import Display  # pyright:ignore[reportPrivateImportUsage]
from stable_baselines3 import PPO
from stable_baselines3.common.base_class import BaseAlgorithm
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.vec_env import VecEnv, VecMonitor
from supersuit.vector.concat_vec_env import ConcatVecEnv
from ursinaxball import Game
from ursinaxball.common_values import ActionBin, BaseMap, GameState, TeamID
from ursinaxball.modules import Bot, GameScore, PlayerHandler
from ursinaxball.modules.bots import ChaseBot, GoalkeeperBot, RandomBot
from ursinaxball.modules.bots.advanced_bots import (
    follow_point,
    position_keeper,
    shoot_disc_close,
)
from ursinaxball.objects.base.disc_object import Disc

warnings.filterwarnings("ignore", category=DeprecationWarning)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
# @title Define Constants
FOLDER_REC = "recordings/"
FOLDER_MODELS = "models/"
TENSORBOARD_LOG = "tensorboard_logs/"
FOLDER_MODELS_ID = "1NdFI8VP5vcev2Q7cSfVdgb0E827F4uvq"
FOLDER_TENSORBOARD_LOG_ID = "1l8JRQbgrx7JIQX9QWfNEUDIzU1BJ_R0K"
TICK_SKIP = 5
N_ENVS = multiprocessing.cpu_count()

!mkdir -p recordings/ models/ tensorboard_logs/

## Define Bots

In [ ]:
# @title Define Common Bots
BOTS = {
    "RandomBot": RandomBot,
    "ChaseBot": ChaseBot,
    "GoalkeeperBot": GoalkeeperBot,
}

In [ ]:
# @title Define Custom Bots
class StrikerBot(Bot):
    """
    This bot chases the ball, positions itself behind it,
    and shoots towards the opposing goal.
    """

    def __init__(self, tick_skip: int = TICK_SKIP):
        super().__init__(tick_skip=tick_skip)
        self.previous_actions: list[int] = [0, 0, 0]

    def step_game(self, player: PlayerHandler, game: Game) -> list[int]:
        ball = game.stadium_game.discs[0]

        # Determine the target goal
        target_goal = game.stadium_game.goals[1 if player.team == TeamID.RED
            else 0]

        # Calculate the center of the opposing goal
        goal_center = np.array(
            (
                (target_goal.points[0][0] + target_goal.points[1][0]) / 2,
                (target_goal.points[0][1] + target_goal.points[1][1]) / 2,
            )
        )

        # 1. Calculate the vector from the goal to the ball
        vec_goal_to_ball = ball.position - goal_center
        vec_goal_to_ball_unit = vec_goal_to_ball / np.linalg.norm(vec_goal_to_ball)

        # 2. Set a target point slightly behind the ball
        offset = ball.radius + player.disc.radius
        strike_target = ball.position + (vec_goal_to_ball_unit * offset)

        # 3. Generate movement and kick actions
        inputs_player = follow_point(player, strike_target.tolist(), 2)

        # 4. Kick if near the target point
        inputs_player[ActionBin.KICK] = shoot_disc_close(
            player, ball, 15, self.previous_actions
        )

        self.previous_actions = inputs_player
        return inputs_player


BOTS["StrikerBot"] = StrikerBot


class AllRounderBot(StrikerBot):
    """
    This bot chases the ball, shoots it towards the goal, defends goals, and passes to teammates.
    """

    def __init__(self, tick_skip: int = TICK_SKIP):
        super().__init__(tick_skip=tick_skip)
        self.previous_actions: list[int] = [0, 0, 0]

    def step_game(self, player: PlayerHandler, game: Game) -> list[int]:
        ball = game.stadium_game.discs[0]

        # Get teammates
        teammates = []
        for plr in game.players:
            if plr.team == player.team and plr != player:
                teammates.append(plr)

        # Determine the target goal
        target_goal = game.stadium_game.goals[1 if player.team == TeamID.RED
            else 0]

        # Calculate the center of the opposing goal
        goal_center = np.array(
            (
                (target_goal.points[0][0] + target_goal.points[1][0]) / 2,
                (target_goal.points[0][1] + target_goal.points[1][1]) / 2,
            )
        )

        # Calculate the vector from the goal to the ball
        vec_goal_to_ball = ball.position - goal_center
        vec_goal_to_ball_unit = vec_goal_to_ball / np.linalg.norm(vec_goal_to_ball)

        # Set a target point slightly behind the ball
        offset = ball.radius + player.disc.radius
        strike_target = ball.position + (vec_goal_to_ball_unit * offset)

        # Change target point to face towards teammate if they are closer to goal
        closest_teammate_to_goal = player
        for teammate in teammates:

            teammate_dist_to_goal = np.abs(teammate.disc.position - goal_center)
            player_dist_to_goal = np.abs(
                closest_teammate_to_goal.disc.position - goal_center
            )

            if teammate_dist_to_goal < player_dist_to_goal:
                closest_teammate_to_goal = teammate
                vec_teammate_to_ball = ball.position - teammate.disc.position
                vec_teammate_to_ball_unit = vec_teammate_to_ball / np.linalg.norm(
                    vec_teammate_to_ball
                )
                strike_target = ball.position + (vec_teammate_to_ball_unit * offset)

        # Generate movement and kick actions
        inputs_player = follow_point(player, strike_target.tolist(), 2)

        # Kick if near the target point
        dist_to_target = np.abs(player.disc.position - strike_target)
        inputs_player[ActionBin.KICK] = shoot_disc_close(
            player, ball, 15, self.previous_actions
        )

        self.previous_actions = inputs_player
        return inputs_player

## Define Environment

In [3]:
# @title Define Game
DEFAULT_GAME = Game(
    stadium_file=BaseMap.CLASSIC,
    enable_renderer=False,
    enable_recorder=False,
)

In [ ]:
# @title Define Terminal Conditions

DEFAULT_TERMINAL_CONDITIONS: tuple[TerminalCondition, ...] = (
    common_conditions.TimeoutCondition(300),
    common_conditions.GoalScoredCondition(),
    common_conditions.NoTouchTimeoutCondition(150),
)

In [ ]:
# @title Define Reward Function
DEFAULT_REWARD_FN = CombinedReward(
    (
        EventReward(
            team_goal=10.0,
            team_concede=-10.0,
            touch=0.0,
            kick=0.1,
        ),
        VelocityPlayerToBallReward(),
        VelocityBallToGoalReward(DEFAULT_GAME.stadium_game),
    ),
    reward_weights=(1.0, 0.1, 1.0),
)

In [ ]:
# @title Define Observation Builder
class EgocentricObs(DefaultObs):
    """
    Builds an egocentric observation space for the agent.
    All positions (ball, allies, enemies) are normalized relative to the agent's current position.

    Observation Array Shape (Length: 11 + 4 * (allies + enemies)):
    Indices:
    [0:4]   - ball.x, ball.y, ball.vx, ball.vy (relative to self)
    [4]     - game.state (0: Kickoff, 1: Playing)
    [5]     - game.team_kickoff (1: Own team kicking off, 0: Other team kicking off)
    [6]     - self.is_kicking (1: True, 0: False)
    [7:11]  - self.x, self.y, self.vx, self.vy (absolute normalized positions)

    Dynamic Indices (Based on team sizes):
    [11:...] - (ally.x, ally.y, ally.vx, ally.vy) for ally in allies (relative to self)
    [...:...] - (enemy.x, enemy.y, enemy.vx, enemy.vy) for enemy in enemies (relative to self)
    """

    MAX_VEL = 15.0

    def build_obs(
        self,
        player: PlayerHandler,
        state: HBGGameState,
        previous_action: npt.NDArray[np.int_],
    ) -> npt.NDArray[np.float64]:
        obs: npt.NDArray[np.float64] = super().build_obs(player, state, previous_action)

        width = state.stadium_game.width
        height = state.stadium_game.height

        self_x, self_y = obs[7] / width, obs[8] / height

        def norm(mvt: npt.NDArray[np.float64]):
            return mvt / np.array(
                [width, height, EgocentricObs.MAX_VEL, EgocentricObs.MAX_VEL]
            ) - np.array([self_x, self_y, 0, 0])

        obs[0:4] = norm(obs[0:4])
        for i in range(7, 7 + 4 * len(state.players), 4):
            obs[i : i + 4] = norm(obs[i : i + 4])
        obs[7] += self_x
        obs[8] += self_y

        obs[4] = state.state  # Kickoff / Playing
        obs[5] = player.team == state.team_kickoff
        obs[6] = player.is_kicking()

        # Sort allies and enemies by distance to ball
        allies = len([p for p in state.players if p.team == player.team]) - 1
        enemies = len(state.players) - allies - 1
        allies_obs = [obs[i : i + 4] for i in range(11, 11 + 4 * allies, 4)]
        enemies_obs = [
            obs[i : i + 4]
            for i in range(11 + 4 * allies, 11 + 4 * allies + 4 * enemies, 4)
        ]
        allies_obs.sort(key=lambda x: np.linalg.norm(x[0:2] - obs[0:2]))
        enemies_obs.sort(key=lambda x: np.linalg.norm(x[0:2] - obs[0:2]))
        obs[11 : 11 + 4 * allies + 4 * enemies] = np.concatenate(
            allies_obs + enemies_obs
        )

        return obs


DEFAULT_OBS_BUILDER = EgocentricObs()

In [ ]:
# @title Define State Setter
class CombinedState(StateSetter):
    def __init__(
        self,
        state_setters: tuple[StateSetter, ...],
        state_weights: tuple[float, ...] | None = None,
    ):
        super().__init__()
        self.state_setters = state_setters
        if state_weights is None:
            self.state_weights = np.ones(len(state_setters), dtype=float)
        else:
            self.state_weights = np.array(state_weights, dtype=float)
        self.state_weights /= self.state_weights.sum()
        self._rng = np.random.default_rng()

    def reset(self, game: Game, save_recording: bool):
        idx = self._rng.choice(len(self.state_weights), p=self.state_weights)
        self.state_setters[idx].reset(game, save_recording)


class RandomKickoff(DefaultState):
    def __init__(self, red_percent=0.5):
        super().__init__()
        self.red_percent = red_percent
        self._rng = np.random.default_rng()

    def reset(self, game: Game, save_recording: bool):
        super().reset(game, save_recording)
        game.state = GameState.KICKOFF
        game.team_kickoff = (
            TeamID.RED if self._rng.random() < self.red_percent else TeamID.BLUE
        )


class StrikerDrill(RandomState):
    def reset(self, game: Game, save_recording: bool):
        super().reset(game, save_recording)

        ball, *placed = game.stadium_game.discs
        placed.extend(player.disc for player in game.players)
        if (game.team_kickoff == TeamID.RED) == (ball.position[0] < 0):
            ball.position[0] *= -1
            for player in game.players:
                player.disc.position[0] *= -1
        ball.velocity /= 5

        for player in game.players:
            if player.team == game.team_kickoff:
                pos = player.disc.position
                radius = player.disc.radius
                offset = np.array(
                    [
                        ball.radius + radius + self._rng.uniform(0, radius),
                        self._rng.uniform(-2 * radius, 2 * radius),
                    ],
                    dtype=float,
                )
                offset[0] *= -1 if player.team == TeamID.RED else 1
                player.disc.position = ball.position + offset
                if not self.is_valid_position(player.disc, placed):
                    player.disc.position = pos
                break


class GoalkeeperDrill(RandomState):
    def reset(self, game: Game, save_recording: bool):
        super().reset(game, save_recording)

        ball, *placed = game.stadium_game.discs
        placed.extend(player.disc for player in game.players)
        if (game.team_kickoff == TeamID.RED) == (ball.position[0] > 0):
            ball.position[0] *= -1
            for player in game.players:
                player.disc.position[0] *= -1

        # Determine the team goal
        goal = (
            game.stadium_game.goals[0]
            if player.team == TeamID.RED
            else game.stadium_game.goals[1]
        )
        # Calculate the target in the team goal
        goal_target = goal.points[0] + self._rng.random() * (goal.points[1] - goal.points[0])
        ball.velocity = (goal_target - ball.position) / 10 + 1

        for player in game.players:
            if player.team == game.team_kickoff:
                pos = player.disc.position
                radius = player.disc.radius
                offset = np.array(
                    [
                        ball.radius + radius + self._rng.uniform(0, radius),
                        self._rng.uniform(-2 * radius, 2 * radius),
                    ],
                    dtype=float,
                )
                offset[0] *= -1 if player.team == TeamID.RED else 1
                player.disc.position = ball.position + offset
                if not self.is_valid_position(player.disc, placed):
                    player.disc.position = pos
                break



class PassingDrill(RandomState):
    pass

DEFAULT_STATE_SETTER = CombinedState(
    (
        RandomKickoff(red_percent=0.7),
        StrikerDrill(red_percent=0.7),
    ),
    state_weights=(0.5, 0.5),
)

In [ ]:
# @title Define Haxball Parallel Environment
class HaxballParallelEnv(ParallelEnv):
    """
    Wraps the standard HaxballGym environment into a PettingZoo Parallel Environment for multi-agent reinforcement learning.
    """
    metadata = {"render_modes": [], "name": "haxball_pe_v1"}

    def __init__(self, **make_kwargs):
        self.render_mode = None
        self.gym_env = haxballgym.make(**make_kwargs)
        self.team_size = int(make_kwargs.get("team_size", 1))
        bots = make_kwargs.get("bots", None)

        self.possible_agents: list[str] = [f"red_{i}" for i in range(self.team_size)]

        if bots is None:
            self.possible_agents += [f"blue_{i}" for i in range(self.team_size)]

        self.agents: list[str] = self.possible_agents[:]
        self.observation_spaces = {
            agent: self.gym_env.observation_space for agent in self.possible_agents
        }
        self.action_spaces = {
            agent: self.gym_env.action_space for agent in self.possible_agents
        }

    def step(self, actions_dict: dict):
        obs, reward, terminated, truncated, info = self.gym_env.step(
            list(actions_dict.values())
        )
        if len(self.agents) == 1:
            obs = [obs]
            reward = [reward]

        observations = dict(zip(self.agents, obs))
        rewards = dict(zip(self.agents, reward))
        terminations = dict.fromkeys(self.agents, terminated)
        truncations = dict.fromkeys(self.agents, truncated)
        infos = dict.fromkeys(self.agents, info)

        if terminated or truncated:
            self.agents = []

        return observations, rewards, terminations, truncations, infos

    def reset(self, seed=None, options=None):
        self.agents = self.possible_agents[:]

        obs, info = self.gym_env.reset(seed=seed, options=options)
        if len(self.agents) == 1:
            obs = [obs]

        observations = dict(zip(self.agents, obs))
        infos = dict.fromkeys(self.agents, info)

        return observations, infos

    def observation_space(self, agent):
        return self.observation_spaces[agent]

    def action_space(self, agent):
        return self.action_spaces[agent]

    def close(self):
        return self.gym_env.close()

In [ ]:
# @title Define Make Environment
if not hasattr(ConcatVecEnv, "seed"):
    def seed_dummy(self, seed=None):
        return [None] * self.num_envs
    ConcatVecEnv.seed = seed_dummy  # pyright:ignore[reportAttributeAccessIssue]


def make_env(
    game: Game = DEFAULT_GAME,
    tick_skip=TICK_SKIP,
    team_size=1,
    terminal_conditions: tuple[TerminalCondition, ...] = DEFAULT_TERMINAL_CONDITIONS,
    bots: list[Bot] | None = [],
    reward_fn: RewardFunction = DEFAULT_REWARD_FN,
    obs_builder: ObsBuilder = DEFAULT_OBS_BUILDER,
    state_setter: StateSetter = DEFAULT_STATE_SETTER,
) -> ParallelEnv:
    env = HaxballParallelEnv(
        game=copy.deepcopy(game),
        tick_skip=tick_skip,
        team_size=team_size,
        bots=bots,
        reward_fn=reward_fn,
        terminal_conditions=terminal_conditions,
        obs_builder=obs_builder,
        state_setter=state_setter,
    )
    env = ss.pettingzoo_env_to_vec_env_v1(env)
    env = ss.concat_vec_envs_v1(
        env, num_vec_envs=N_ENVS, num_cpus=0, base_class="stable_baselines3"
    )

    return VecMonitor(env)

## Load Models

In [ ]:
# @title Define ModelBot Wrapper
def ModelBot(
    model: BaseAlgorithm,
    trained_allies: int = 0,
    obs_builder: ObsBuilder = DEFAULT_OBS_BUILDER,
):
    """
    Wraps a trained Stable-Baselines3 model so it can be used as a standard Bot in the game.
    Handles padding or truncating the observation array if the bot is placed in a match with a different number of allies/enemies than it was trained for.
    """
    obs_len = (model.observation_space.shape or (11,))[0]
    trained_enemies = (obs_len - 11) // 4 - trained_allies

    class _ModelBot(Bot):
        MODEL = model

        def __init__(self, tick_skip=TICK_SKIP):
            super().__init__(tick_skip=tick_skip)

        def step_game(
            self, _player: PlayerHandler, _game: Game
        ) -> npt.NDArray[np.int_]:
            obs = obs_builder.build_obs(
                _player, HBGGameState(_game), np.zeros(3, dtype=float)
            )
            obs_norm = obs[:11]

            team = _player.team
            allies = len([p for p in _game.players if p.team == team]) - 1
            enemies = len(_game.players) - allies - 1

            if trained_allies <= allies:
                obs_norm = np.concatenate(
                    [
                        obs_norm,
                        obs[11 : 11 + 4 * trained_allies],
                    ]
                )
            else:
                obs_norm = np.concatenate(
                    [
                        obs_norm,
                        obs[11 : 11 + 4 * allies],
                        np.array([0, 0, 0, 0]).repeat(trained_allies - allies),
                    ]
                )

            if trained_enemies <= enemies:
                obs_norm = np.concatenate(
                    [
                        obs_norm,
                        obs[11 + 4 * allies : 11 + 4 * allies + 4 * trained_enemies],
                    ]
                )
            else:
                obs_norm = np.concatenate(
                    [
                        obs_norm,
                        obs[11 + 4 * allies :],
                        np.array([0, 0, 0, 0]).repeat(trained_enemies - enemies),
                    ]
                )

            action, _states = self.MODEL.predict(obs_norm, deterministic=True)
            action -= np.array([1, 1, 0], dtype=int)
            action[0] *= 1 if team == TeamID.RED else -1
            return action

    return _ModelBot

In [ ]:
# @title Define Drive
from pathlib import Path

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from pydrive2.files import GoogleDriveFile

try:
    from google.colab import auth
    from oauth2client.client import GoogleCredentials

    auth.authenticate_user()
    gauth = GoogleAuth()
    gauth.credentials = GoogleCredentials.get_application_default()
except:
    if Path("client_secrets.json").exists():
        with open("client_secrets.json", "r") as f:
            client_secrets = f.read()
    else:
        with open("client_secrets.json", "w") as f:
            f.write(input("client_secrets.json: "))
        gauth = GoogleAuth()
        gauth.settings["get_refresh_token"] = True
        gauth.CommandLineAuth()

drive = GoogleDrive(gauth)

In [ ]:
# @title Load Drive Files
def load_model(model_file: GoogleDriveFile, allies: int | None = None):
    file_name = model_file.get("title") or ""
    algorithm, matchup = file_name.split("-", 1)
    if allies is None:
        allies = int(matchup[0]) - 1
    model_path = f"{FOLDER_MODELS}/{file_name}"
    model_file.GetContentFile(model_path)
    match algorithm:
        case "ppo":
            BOTS[file_name.rstrip(".zip")] = ModelBot(PPO.load(model_path), allies)
            print(f"Loaded PPO {file_name}")


def load_log(log_file: GoogleDriveFile):
    file_name = log_file.get("title") or ""
    log_path = f"{TENSORBOARD_LOG}/{file_name}"
    log_file.GetContentFile(log_path)
    shutil.unpack_archive(log_path, TENSORBOARD_LOG)
    print(f"Loaded Log {file_name}")


models = drive.ListFile(
    {
        "q": f"'{FOLDER_MODELS_ID}' in parents and mimeType='application/zip' and trashed=false"
    }
).GetList()
for model_file in models:
    load_model(model_file)

logs = drive.ListFile(
    {
        "q": f"'{FOLDER_TENSORBOARD_LOG_ID}' in parents and mimeType='application/zip' and trashed=false"
    }
).GetList()
for log_file in logs:
    load_log(log_file)

In [ ]:
# @title Save Files
def save_model_path(model_path: str) -> GoogleDriveFile:
    model_name = Path(model_path).name
    model_file = drive.CreateFile(
        {"title": model_name, "parents": [{"id": FOLDER_MODELS_ID}]}
    )
    model_file.SetContentFile(model_path)
    model_file.Upload()
    return model_file


def save_model(model: BaseAlgorithm, model_name: str) -> GoogleDriveFile:
    model_path = f"{FOLDER_MODELS}/{model_name}.zip"
    model.save(model_path)
    return save_model_path(model_path)


def save_logs_path(model_name: str) -> GoogleDriveFile:
    log_path = f"{TENSORBOARD_LOG}/{model_name}_0"
    shutil.make_archive(log_path, "zip", root_dir=TENSORBOARD_LOG, base_dir=model_name)
    log_file = drive.CreateFile(
        {"title": f"{model_name}.zip", "parents": [{"id": FOLDER_TENSORBOARD_LOG_ID}]}
    )
    log_file.SetContentFile(f"{log_path}.zip")
    log_file.Upload()
    print(f"Saved Log {model_name}.zip")
    return log_file


def save_logs(model: BaseAlgorithm) -> GoogleDriveFile:
    log_path = Path(str(model.logger.dir))
    return save_logs_path(log_path.name)

## Define Training

In [ ]:
# @title Define Train Model
class SaveModelCheckpointCallback(CheckpointCallback):
    def _on_step(self) -> bool:
        continue_training = super()._on_step()
        if self.n_calls % self.save_freq == 0:
            model_path = self._checkpoint_path(extension="zip")
            save_model_path(model_path)
            save_logs_path(f"{self.name_prefix}_0")
        return continue_training


def train_model(
    model: BaseAlgorithm,
    model_name: str,
    total_timesteps: int,
    save_freq: int = 65_536,
    progress_bar: bool = True,
):
    checkpoint_callback = SaveModelCheckpointCallback(
        save_freq=save_freq,
        save_path=f"{FOLDER_MODELS}/{model_name}",
        name_prefix=model_name,
    )
    model.learn(
        total_timesteps=total_timesteps,
        tb_log_name=model_name,
        reset_num_timesteps=False,
        progress_bar=progress_bar,
        callback=checkpoint_callback,
    )
    model.env.close()  # pyright:ignore[reportOptionalMemberAccess]
    save_model(model, model_name)
    save_logs(model)

In [ ]:
# @title Define Curriculum
def train_curriculum(
    model: BaseAlgorithm,
    model_name: str,
    envs: list[ParallelEnv],
    total_timesteps: list[int],
    save_freq: int = 65_536,
    progress_bar: bool = True,
):
    model.tensorboard_log = TENSORBOARD_LOG
    for i, (env, timesteps) in enumerate(zip(envs, total_timesteps)):
        model.env = env
        model.env.reset()
        train_model(
            model,
            model_name=f"{model_name}_drill-{i}",
            total_timesteps=timesteps,
            save_freq=save_freq,
        )
        model.env.close()

## Train Models

[models - Google Drive](https://drive.google.com/drive/folders/1NdFI8VP5vcev2Q7cSfVdgb0E827F4uvq)

Bypass Colab Timeout:
- Run in console:
```js
setInterval(document.querySelector("#top-toolbar > colab-connect-button").shadowRoot.querySelector("#connect").click,30000)
```
- Save bookmark:
```
javascript:(setInterval(document.querySelector("#top-toolbar > colab-connect-button").shadowRoot.querySelector("#connect").click,30000))
```


In [ ]:
# @title Start Tensorboard
%load_ext tensorboard
%tensorboard --logdir /content/tensorboard_logs

In [ ]:
# @title Train PPO 1v0
model_name = "ppo-1v0_v0"
if model_name not in BOTS.keys():
    model = PPO(
        "MlpPolicy",
        make_env(team_size=1, bots=[]),
        tensorboard_log=TENSORBOARD_LOG,
    )
    envs: list[ParallelEnv] = [
        make_env(
            team_size=1,
            bots=[],
            state_setter=StrikerDrill(red_percent=1),
        ),
        make_env(
            team_size=1,
            bots=[],
            state_setter=GoalkeeperDrill(red_percent=1),
        ),
        make_env(
            team_size=1,
            bots=[],
            state_setter=DEFAULT_STATE_SETTER,
        ),
    ]
    train_curriculum(
        model,
        model_name=model_name,
        envs=envs,
        total_timesteps=[16_384, 16_384, 16_384],
        save_freq=16_384,
        # total_timesteps=1_048_576,
        # save_freq=65_536,
    )
    BOTS[model_name] = ModelBot(model, 0)

In [ ]:
# @title Train PPO 1v1
model_name = "ppo-1v1_v1"
if model_name not in BOTS.keys():
    model = PPO(
        "MlpPolicy",
        make_env(team_size=1, bots=[BOTS["ChaseBot"](TICK_SKIP)]),
        tensorboard_log=TENSORBOARD_LOG,
    )
    train_model(
        model,
        model_name=model_name,
        total_timesteps=1_048_576,
        save_freq=65_536,
        # total_timesteps=65_536,
        # save_freq=16_384,
        # progress_bar=False,
    )
    BOTS[model_name] = ModelBot(model, 0)

In [ ]:
# @title Train PPO 1v1 Self-Play
model_name = "ppo-1v1-selfplay_v1"
if model_name not in BOTS.keys():
    model = PPO(
        "MlpPolicy",
        make_env(team_size=1, bots=None),
        tensorboard_log=TENSORBOARD_LOG,
    )
    model = train_model(
        model,
        model_name=model_name,
        # total_timesteps=1_048_576,
        # save_freq=65_536,
        total_timesteps=65_536,
        save_freq=16_384,
        # progress_bar=False,
    )
    BOTS[model_name] = ModelBot(model, 0)

In [ ]:
model_name = "ppo-1v1-selfplay-test_v0"
model = PPO(
    "MlpPolicy",
    make_env(
        team_size=1,
        bots=None,
        state_setter = StrikerDrill()
    ),
    tensorboard_log=TENSORBOARD_LOG,
)
train_model(
    model,
    model_name=model_name,
    total_timesteps=16_384,
    save_freq=16_384,
)

In [ ]:
model_name = "ppo-1v1-selfplay-test_v0"
model = BOTS[model_name].MODEL
model.env = make_env(
    team_size=1,
    bots=None,
    state_setter = StrikerDrill()
)
model.env.reset()
model.tensorboard_log = TENSORBOARD_LOG
train_model(
    model,
    model_name=model_name,
    total_timesteps=16_384,
    save_freq=16_384,
    progress_bar=False,
)

In [ ]:
# @title Train PPO 1v1 Self-Play Drills
model_name = "ppo-1v1-selfplay-drills_v1"
model = PPO(
    "MlpPolicy",
    make_env(
        team_size=1,
        bots=None,
        state_setter = RandomKickoff()
    ),
    tensorboard_log=TENSORBOARD_LOG,
)
train_model(
    model,
    model_name=model_name,
    total_timesteps=65_536,
    save_freq=16_384,
)
model.env = make_env(
    team_size=1,
    bots=None,
    state_setter = StrikerDrill()
)
train_model(
    model,
    model_name=model_name,
    total_timesteps=65_536,
    save_freq=16_384,
)

In [ ]:
# @title Train PPO 2v2 Self-Play
model_name = "ppo-2v2-selfplay_v1"
if model_name not in BOTS.keys():
    model = PPO(
        "MlpPolicy",
        make_env(team_size=2, bots=None),
        tensorboard_log=TENSORBOARD_LOG,
    )
    train_model(
        model,
        model_name=model_name,
        total_timesteps=1_048_576,
        save_freq=65_536,
        # total_timesteps=65_536,
        # save_freq=16_384,
        # progress_bar=False,
    )
    BOTS[model_name] = ModelBot(model, 1)

In [ ]:
# @title Train PPO 3v3 Self-Play
model_name = "ppo-3v3-selfplay_v1"
if model_name not in BOTS.keys():
    model = PPO(
        "MlpPolicy",
        make_env(team_size=3, bots=None),
        tensorboard_log=TENSORBOARD_LOG,
    )
    train_model(
        model,
        model_name=model_name,
        total_timesteps=1_048_576,
        save_freq=65_536,
        # total_timesteps=65_536,
        # save_freq=16_384,
        # progress_bar=False,
    )
    BOTS[model_name] = ModelBot(model, 2)

In [ ]:
# @title Gym Debug
# https://wazarr94.github.io/
game = Game(folder_rec=FOLDER_REC, enable_renderer=False)
env = haxballgym.make(
    game=game,
    tick_skip=TICK_SKIP,
    # team_size=1,
    team_size=3,
    # bots=[],
    # bots=[BOTS["ChaseBot"](1)],
    bots=[
        BOTS["RandomBot"](1),
        BOTS["ChaseBot"](1),
        BOTS["GoalkeeperBot"](1),
    ],
    reward_fn=DEFAULT_REWARD_FN,
    obs_builder=DEFAULT_OBS_BUILDER,
    state_setter=RandomKickoff(red_percent=1.),
)

# model = BOTS["ppo-1v1-selfplay_v1"].MODEL
# model = BOTS["ppo-2v2-selfplay_v1"].MODEL
model = BOTS["ppo-3v3-selfplay_v1"].MODEL

ep_reward = 0
obs, info = env.reset()
# for step in range(20):
for step in range(20_000):
    # print(step, obs)
    # actions, _states = model.predict(obs)
    actions = [model.predict(o)[0] for o in obs]
    # obs, rewards, terminated, truncated, info = env.step(action)
    obs, rewards, terminated, truncated, info = env.step(actions)
    # ep_reward += rewards
    # print(step, actions, rewards)
    if terminated or truncated:
        break
obs, info = env.reset(options={"save_recording": True})
print(ep_reward)

## Teams

In [ ]:
# @title Define Teams
Team = dict[str, int]
Teams = dict[str, Team]

TEAMS_1: Teams = dict([(f"{bot}-1", {bot: 1}) for bot in BOTS])
TEAMS_2: Teams = dict([(f"{bot}-2", {bot: 2}) for bot in BOTS])
TEAMS_3: Teams = dict([(f"{bot}-3", {bot: 3}) for bot in BOTS])
TEAMS_3.update([(f"GoalkeeperBot-1_StrikerBot-2", {"GoalkeeperBot": 1, "StrikerBot": 2})])

TEAMS: Teams = {**TEAMS_1, **TEAMS_2, **TEAMS_3}
TEAMS["Empty-0"] = {}

print("\n".join(TEAMS.keys()))

## Evaluation

In [ ]:
# @title Run Game
def run_game(
    red_team: Team | str,
    blue_team: Team | str,
    *,
    step_limit: int = 2_000,
    score_limit: int = 3,
    enable_renderer: bool = False,
    enable_recorder: bool = False,
    folder_rec: str = "",
) -> GameScore:
    if not folder_rec and isinstance(red_team, str) and isinstance(blue_team, str):
        folder_rec = f"{red_team}_vs_{blue_team}"
    folder_rec = f"{FOLDER_REC}/{folder_rec}"

    if isinstance(red_team, str):
        red_team = TEAMS[red_team]
    if isinstance(blue_team, str):
        blue_team = TEAMS[blue_team]

    game = Game(
        folder_rec=folder_rec,
        enable_renderer=enable_renderer,
        fov=420,
        enable_recorder=enable_recorder,
    )
    game.score = GameScore(score_limit=score_limit)

    for bot, count in red_team.items():
        for i in range(count):
            game.add_player(
                PlayerHandler(f"{bot}_{i}", TeamID.RED, BOTS[bot](tick_skip=1))
            )
    for bot, count in blue_team.items():
        for i in range(count):
            game.add_player(
                PlayerHandler(f"{bot}_{i}", TeamID.BLUE, BOTS[bot](tick_skip=1))
            )

    game.start()
    for _ in range(step_limit):
        if game.step([player.step(game) for player in game.players]):
            break

    score = copy.deepcopy(game.score)
    game.stop(save_recording=enable_recorder)
    return score


red_team = "ppo-1v1-selfplay_v1-1"
# red_team = "ChaseBot-1"
# red_team = "GoalkeeperBot-1_StrikerBot-2"
# red_team = "ppo-1v1-selfplay_v1_262144_steps-1"
# red_team = "ppo-2v2-selfplay_v1-2"
blue_team = "ppo-1v1-selfplay_v1-1"
# blue_team = "GoalkeeperBot-1"
# blue_team = "ChaseBot-1"
# blue_team = "ChaseBot-2"
# blue_team = "ppo-1v1-selfplay_v1-1"
# blue_team = "ppo-2v2-selfplay_v1-2"
score = run_game(
    red_team,
    blue_team,
    step_limit=5_000,
    # score_limit = 2,
    score_limit = 5,
    enable_recorder=True
)
print(score.get_score_string())

In [ ]:
# @title Record Game
def record_game(
    red_team: Team | str,
    blue_team: Team | str,
    filename: str = "",
    size: tuple[int, int] = (420, 200),
    **config,
) -> tuple[GameScore, Video]:
    if not filename and isinstance(red_team, str) and isinstance(blue_team, str):
        filename = f"{red_team}_vs_{blue_team}"
    filename = f"{FOLDER_REC}/{filename}.mp4"

    display = Display(visible=False, size=size)
    display.start()

    ffmpeg_process = subprocess.Popen(
        [
            "ffmpeg",
            "-y",
            "-f",
            "x11grab",
            "-draw_mouse",
            "0",
            "-r",
            "30",
            "-s",
            f"{size[0]}x{size[1]}",
            "-i",
            display.new_display_var,
            # "-preset",
            # "ultrafast",
            filename,
        ]
    )

    queue = multiprocessing.Queue()
    process = multiprocessing.Process(
        target=lambda: queue.put(
            run_game(
                red_team,
                blue_team,
                enable_renderer=True,
                **config,
            )
        )
    )
    process.start()
    process.join()

    ffmpeg_process.terminate()
    try:
        ffmpeg_process.wait(timeout=2)
    except subprocess.TimeoutExpired:
        ffmpeg_process.kill()
    display.stop()
    if process.is_alive():
        process.terminate()

    return queue.get(), Video(filename, embed=True)


# red_team = {'GoalkeeperBot': 1, 'ChaseBot': 1}
# red_team = "ppo-1v1_v1-1"
red_team = "ppo-2v2_v1-1"
blue_team = {'StrikerBot': 1, 'RandomBot': 1}
# blue_team = "Empty-0"
# blue_team = "ChaseBot-1"
score, video = record_game(
    red_team,
    blue_team,
    filename="Game1",
    size=(420, 200),
    # size=(840, 400),
    step_limit=2_000,
)
video

In [ ]:
# @title Run Tournament
def run_tournament(teams: Teams = TEAMS, **config) -> dict[str, PlackettLuceRating]:
    model = PlackettLuce()
    ratings = {name: model.rating(name=name) for name in teams}

    matchups = [(red, blue) for red in teams for blue in teams if red != blue]
    print(f"Running {len(matchups)} matchups...")

    with ProcessPoolExecutor() as executor:
        future_to_match = {
            executor.submit(
                run_game,
                teams[red],
                teams[blue],
                **config,
            ): (red, blue)
            for red, blue in matchups
        }

        for future in future_to_match:
            red, blue = future_to_match[future]
            score = future.result()
            print(f"{red} vs {blue}: {score.red} - {score.blue}")

            [ratings[red]], [ratings[blue]] = model.rate(
                [[ratings[red]], [ratings[blue]]], scores=[score.red, score.blue]
            )

    return ratings


ratings = run_tournament(
    TEAMS_2,
    step_limit=10_000,
    score_limit=5,
)

for name, rating in sorted(ratings.items(), key=lambda x: x[1].ordinal(), reverse=True):
    print(f"{name}: {rating.ordinal()}")